In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Odhad energie základního stavu Heisenbergova řetězce pomocí VQE

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Odhadovaná spotřeba: 37 minut na procesoru Heron (POZNÁMKA: Jedná se pouze o odhad. Tvůj skutečný čas běhu se může lišit.)*
## Výstupy z učení
Po dokončení tohoto tutoriálu bys měl(a) rozumět následujícím informacím:
- Jak modelovat Heisenbergův spinový řetězec jako kvantový hamiltonián pomocí Qiskitu
- Jak používat optimalizátor SPSA k odhadu energie základního stavu kvantového systému
- Jak spouštět variační pracovní postupy na kvantovém hardwaru IBM&reg; pomocí primitiv Qiskit Runtime a relací

## Předpoklady
Doporučujeme se předem seznámit s těmito tématy:
- [Základy kvantových informací](/learning/courses/basics-of-quantum-information)
- [Úvod do Qiskit vzorů](/guides/intro-to-patterns)
- [Návrh variačních algoritmů](/learning/courses/variational-algorithm-design)

## Pozadí
Heisenbergův spinový řetězec je jedním z nejrozšířeněji studovaných modelů v fyzice kondenzované hmoty a kvantovém magnetismu. Popisuje jednorozměrnou mřížku vzájemně působících kvantových spinů, kde sousední spiny jsou vázány prostřednictvím výměnných interakcí. Hamiltonián pro izotropní Heisenbergův model s vnějším magnetickým polem je dán:

$$H = \sum_{\langle i,j \rangle} \left( J_x X_i X_j + J_y Y_i Y_j + J_z Z_i Z_j \right) + \sum_{i} h_i Z_i,$$

kde $X_i$, $Y_i$ a $Z_i$ jsou Pauliho operátory působící na místě $i$, suma $\langle i,j \rangle$ probíhá přes nejbližší sousední páry, $J_x = J_y = J_z = 0.5$ jsou konstanty výměnné vazby (v tomto tutoriálu izotropní) a $h_i$ představuje vnější magnetické pole závislé na místě. V tomto tutoriálu jsou hodnoty magnetického pole náhodně vzorkovány z intervalu $[-1, 1]$. Poznamenejme, že v níže uvedené implementaci je množina párů „nejbližších sousedů" určena nativní vazbou hardwarového backendu pro prvních $N$ qubitů, která nemusí tvořit přísný lineární řetězec v závislosti na topologii zařízení.

Porozumění energii základního stavu tohoto hamiltoniánu má v fyzice zásadní význam. Základní stav zakódovává informace o kvantových fázových přechodech, struktuře entanglementu a magnetickém uspořádání. Klasicky se výpočet přesné energie základního stavu stává nesouditelným s rostoucím počtem spinů, protože dimenze Hilbertova prostoru roste exponenciálně jako $2^N$ pro $N$ spinů. To z něj činí přirozený kandidát pro kvantovou simulaci.

Variační kvantový eigensolver (VQE) je hybridní kvantově-klasický algoritmus navržený k odhadu energie základního stavu hamiltoniánu. Funguje tak, že na kvantovém počítači připraví parametrizovaný kvantový stav $|\psi(\theta)\rangle$ (zvaný ansatz) a změří střední hodnotu $\langle \psi(\theta) | H | \psi(\theta) \rangle$. Klasický optimalizátor pak iterativně upravuje parametry $\theta$ tak, aby tuto energii minimalizoval, přičemž využívá variačního principu, který zaručuje, že naměřená energie je vždy horním odhadem skutečné energie základního stavu.

V tomto tutoriálu používáme ansatz `efficient_su2` z knihovny obvodů Qiskitu, který konstruuje vrstvy jednobitových rotací a entanglujících hradel. Optimalizace je prováděna pomocí algoritmu Simultaneous Perturbation Stochastic Approximation (SPSA), který je vhodný pro hlučný kvantový hardware, protože odhaduje gradienty pomocí pouhých dvou vyhodnocení funkce za iteraci bez ohledu na počet parametrů.
## Požadavky
Před zahájením tohoto tutoriálu se ujisti, že máš nainstalováno následující:

* Qiskit SDK v2.0 nebo novější, s podporou [vizualizace](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime v0.44 nebo novější (`pip install qiskit-ibm-runtime`)
## Nastavení

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Sequence

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import BaseEstimatorV2
from qiskit.circuit.library import XGate
from qiskit.circuit.library import efficient_su2
from qiskit.transpiler import PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes.scheduling import (
    ALAPScheduleAnalysis,
    PadDynamicalDecoupling,
)
from qiskit_ibm_runtime import QiskitRuntimeService, Session, EstimatorV2


def visualize_results(results):
    plt.plot(results["cost_history"], lw=2)
    plt.xlabel("Number of function evaluations")
    plt.ylabel("Energy")
    plt.show()

## Příklad v malém měřítku

V této části postupujeme krok za krokem vzorem Qiskit v malém měřítku a vysvětlujeme klíčové komponenty při sestavování pracovního postupu.

### Krok 1: Mapování klasických vstupů na kvantový problém

* Vstup: Počet spinů
* Výstup: Ansatz a hamiltonián modelující Heisenbergův řetězec

Sestrojíme ansatz a hamiltonián, které modelují 10-spinový Heisenbergův řetězec. V tomto kroku sestavíme 10-spinový Heisenbergův hamiltonián na propojovací mapě nejméně vytíženého backendu a připravíme ansatz `efficient_su2`.

In [2]:
num_spins = 10
ansatz = efficient_su2(num_qubits=num_spins, reps=2)

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, min_num_qubits=num_spins, simulator=False
)

coupling = backend.target.build_coupling_map()
reduced_coupling = coupling.reduce(list(range(num_spins)))

edge_list = reduced_coupling.graph.edge_list()
ham_list = []

for edge in edge_list:
    ham_list.append(("ZZ", edge, 0.5))
    ham_list.append(("YY", edge, 0.5))
    ham_list.append(("XX", edge, 0.5))

for qubit in reduced_coupling.physical_qubits:
    ham_list.append(("Z", [qubit], np.random.random() * 2 - 1))

hamiltonian = SparsePauliOp.from_sparse_list(ham_list, num_qubits=num_spins)

ansatz.draw("mpl", style="iqp")

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/7e8d2f10-f1d6-4ec2-bac9-9db23499c9e1-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/spin-chain-vqe/extracted-outputs/7e8d2f10-f1d6-4ec2-bac9-9db23499c9e1-0.avif)

### Krok 2: Optimalizace problému pro spuštění na kvantovém hardwaru
* Vstup: Abstraktní Circuit, observabilní veličina
* Výstup: Cílový Circuit a observabilní veličina, optimalizované pro vybrané QPU

Použijeme funkci `generate_preset_pass_manager` z Qiskitu k automatickému vygenerování optimalizační rutiny pro náš obvod vzhledem k vybranému QPU. Volíme `optimization_level=3`, což poskytuje nejvyšší úroveň optimalizace z přednastavených pass managerů. Také zahrneme plánovací průchody `ALAPScheduleAnalysis` a `PadDynamicalDecoupling` pro potlačení chyb způsobených dekoherencí.

In [3]:
target = backend.target
pm = generate_preset_pass_manager(optimization_level=3, target=target)
pm.scheduling = PassManager(
    [
        ALAPScheduleAnalysis(durations=target.durations()),
        PadDynamicalDecoupling(
            durations=target.durations(),
            dd_sequence=[XGate(), XGate()],
            pulse_alignment=target.pulse_alignment,
        ),
    ]
)
isa_ansatz = pm.run(ansatz)
isa_observable = hamiltonian.apply_layout(isa_ansatz.layout)
isa_ansatz.draw("mpl", scale=0.6, style="iqp", fold=-1, idle_wires=False)

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/a0a5f1c8-5c31-4d9f-ae81-37bd67271d44-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/spin-chain-vqe/extracted-outputs/a0a5f1c8-5c31-4d9f-ae81-37bd67271d44-0.avif)

### Krok 3: Spuštění pomocí Qiskit primitiv
* Vstup: Cílový Circuit a observabilní veličina
* Výstup: Výsledky optimalizace

Minimalizujeme odhadovanou energii základního stavu systému optimalizací parametrů Circuitu. Použijeme primitivum `Estimator` z Qiskit Runtime k vyhodnocení účelové funkce během optimalizace.

Protože jsme v kroku 2 optimalizovali Circuit pro backend, můžeme se vyhnout transpilaci na serveru Runtime nastavením `skip_transpilation=True` a předáním optimalizovaného Circuitu. Pro tuto ukázku spustíme výpočet na QPU pomocí primitiv `qiskit-ibm-runtime`. Pro spuštění s primitivy `qiskit` založenými na stavovém vektoru nahraď blok kódu používající primitiva Qiskit Runtime komentovaným blokem.

V tomto tutoriálu používáme Simultaneous Perturbation Stochastic Approximation (SPSA), což je gradientní optimalizátor. Níže uvádíme stručný úvod do SPSA a kód pro jeho implementaci pomocí Qiskit v2.0.
### Představení SPSA
Simultaneous Perturbation Stochastic Approximation (SPSA) [\[1\]](#references) je optimalizační algoritmus, který aproximuje celý gradientní vektor pomocí pouhých dvou volání funkce v každé iteraci. Nechť $f:\mathbb{R}^p\rightarrow \mathbb{R}$ je účelová funkce s $p$ parametry k optimalizaci a $x_i\in \mathbb{R}^p$ je vektor parametrů v $i$-tém kroku iterace. Pro výpočet gradientu se vytvoří náhodný vektor $\Delta_i$ velikosti $p$, kde každý prvek $\Delta_{ij}$, $\forall$ $j\in {1,2,...,p}$, je rovnoměrně vzorkován z ${-1, 1}$. Každý prvek náhodného vektoru $\Delta_i$ je poté vynásoben malou hodnotou $c_i$, čímž vznikne náhodná perturbace. Gradient je pak odhadnut jako

$$[\nabla f(x_i)]_j \approx \frac{f(x_i + c_i \Delta_i) - f(x_i - c_i \Delta_i)}{2c_i\Delta_{ij}}.$$

Intuitivně, protože se při odhadování gradientu aplikuje náhodná perturbace, lze očekávat toleranci vůči malým odchylkám v přesných hodnotách $f$ způsobeným šumem. SPSA je zejména známé svou robustností vůči šumu a vyžaduje pouze dvě volání hardwaru v každé iteraci. Je proto jedním z nejvíce preferovaných optimalizátorů pro implementaci variačních algoritmů.

V tomto tutoriálu jsou hyperparametry pro $i$-tou iteraci, $a_i$ a $c_i$, vypočteny jako

$$a_i = \frac{a}{(A + i + 1)^\alpha} \quad \text{and} \quad c_i = \frac{c}{(i+1)^\gamma},$$

kde konstantní hodnoty jsou $A = 30$, $\alpha = 0.9$, $a = 0.3$, $c = 0.1$ a $\gamma = 0.4$. Tyto hodnoty jsou převzaty z [\[2\]](#references). Pro dobré výsledky SPSA je nutné vhodné ladění hyperparametrů.

In [4]:
def spsa(
    fun, x0, args=(), A=30, alpha=0.9, a=0.3, c=0.1, gamma=0.4, maxiter=100
):
    nparams = len(x0)
    x = np.copy(x0)

    for i in range(maxiter):
        a_i = a / (A + i + 1) ** alpha
        c_i = c / (i + 1) ** gamma
        delta_i = np.random.choice([-1, 1], nparams)

        # two hardware calls
        eval_1 = fun(x + c_i * delta_i, *args)
        eval_2 = fun(x - c_i * delta_i, *args)

        # compute the gradient and update the parameters
        grad = (eval_1 - eval_2) / (2 * c_i) * np.reciprocal(delta_i)
        x = x - a_i * grad

    return x

In [8]:
def cost_func(
    params: Sequence,
    ansatz: QuantumCircuit,
    hamiltonian: SparsePauliOp,
    estimator: BaseEstimatorV2,
    cost_history_dict: dict,
) -> float:
    """Ground state energy evaluation."""
    energy = (
        estimator.run([(ansatz, hamiltonian, [params])]).result()[0].data.evs
    )

    cost_history_dict["iters"] += 1
    cost_history_dict["prev_vector"] = list(params)
    cost_history_dict["cost_history"].append(float(energy[0]))

    print(
        f"Fx Iters. done: {cost_history_dict['iters']} [Current cost: {round(energy[0], 5)}]",
        end="\r",
    )

    return energy


def solve(x0, isa_ansatz, isa_observable, maxiter=150):
    cost_history_dict = {
        "prev_vector": None,
        "iters": 0,
        "cost_history": [],
        "y_min": None,
    }

    # Evaluate the problem using a QPU via Qiskit IBM Runtime
    with Session(backend=backend) as session:
        estimator = EstimatorV2(mode=session)
        estimator.skip_transpilation = True
        estimator.options.environment.job_tags = ["TUT_HSVQE"]
        x_opt = spsa(
            cost_func,
            x0=x0,
            args=(isa_ansatz, isa_observable, estimator, cost_history_dict),
            maxiter=maxiter,
        )

        y_min = cost_func(
            x_opt, isa_ansatz, isa_observable, estimator, cost_history_dict
        )

    return y_min, cost_history_dict

In [9]:
np.random.seed(42)
num_params = ansatz.num_parameters
params = 2 * np.pi * np.random.random(num_params)

Zde nastavíme `maxiter = 50`. Protože každá iterace vyžaduje dvě volání funkce pro výpočet gradientu, celkový počet volání funkce bude $2 \times \text{maxiter}$. Hodnotu `maxiter` lze zvýšit pro lepší odhad energie.

In [10]:
maxiter = 50
spsa_min, spsa_history = solve(
    params, isa_ansatz, isa_observable, maxiter=maxiter
)

Fx Iters. done: 101 [Current cost: -3.03843]

### Step 4: Post-process and return result in desired classical format

* Input: Ground-state energy estimates during optimization
* Output: Estimated ground-state energy

In [11]:
print(f"Estimated ground state energy: {spsa_min}")

Estimated ground state energy: [-3.03842968]


In [12]:
results = {
    "spsa": spsa_history,
}

visualize_results(spsa_history)

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/ecd7762a-0.avif" alt="Output of the previous code cell" />

## Large-scale hardware example

A large-scale hardware example is not included in this tutorial. As the number of qubits increases, VQE encounters significant challenges due to the [barren plateau](/learning/courses/variational-algorithm-design/optimization-loops#barren-plateaus) phenomenon: the gradient of the cost function vanishes exponentially with system size, making optimization practically infeasible for large circuits. Combined with hardware noise, this means that scaling VQE to larger spin chains does not produce reliably reproducible results. For approaches that overcome these limitations, see the Next Steps section below.

## Challenge

Now that you have a working VQE implementation for the Heisenberg chain, try the following:

1. **Experiment with ansatz depth:** Modify the `reps` parameter in `efficient_su2` (for example, try `reps=1` and `reps=3`). How does ansatz depth affect the estimated ground-state energy and convergence speed? At what point do you observe diminishing returns or instability?
2. **Tune SPSA hyperparameters:** Adjust the learning rate schedule parameters (`a`, `c`, `alpha`, `gamma`, `A`) and observe how they impact convergence. Can you find a configuration that converges faster than the defaults used here?
3. **Compare coupling topologies:** Instead of using the backend's native coupling map, try constructing a simple nearest-neighbor linear chain and compare the results. How does the connectivity of the physical hardware affect the transpiled circuit depth and final energy estimate?

## References

\[1] Spall, J. C. (2002). Implementation of the simultaneous perturbation algorithm for stochastic optimization.
IEEE Transactions on Aerospace and Electronic Systems, 34(3), 817-823.

\[2] Sahin, M. Emre, et al. (2025). Qiskit Machine Learning: an open-source library for quantum machine learning tasks at scale on quantum hardware and classical simulators. arXiv:2505.17756.

## Next steps

<Admonition type="tip" title="Recommendations">
  If you found this work interesting, you might be interested in the following material:

  * **Try Sample-based Quantum Diagonalization (SQD):** As demonstrated in this tutorial, VQE faces challenges at scale due to barren plateaus and high measurement overhead. IBM has developed [Sample-based Quantum Diagonalization (SQD)](/docs/addons/qiskit-addon-sqd) as a more scalable alternative. Unlike VQE, SQD avoids variational optimization entirely; instead, a quantum computer generates samples and a classical computer projects the Hamiltonian onto a subspace spanned by those samples and diagonalizes it. This provides an upper bound to the ground-state energy with significantly fewer measurements and without susceptibility to barren plateaus. Follow the [SQD tutorial](/docs/tutorials/sample-based-quantum-diagonalization) to see this approach in action.
  * **Explore the Quantum Diagonalization Algorithms course:** Deepen your understanding of both VQE and SQD, including their trade-offs, in the [Quantum diagonalization algorithms](/learning/courses/quantum-diagonalization-algorithms) course on IBM Quantum Learning.
</Admonition>